# Análisis Macroeconómico de Colombia

**Autor:** Frederick Salazar Sanchez <br>
**Fecha:** Marzo 2026 <br>
**Descripción:** Análisis descriptivo completo de los indicadores macroeconómicos de Colombia desde 1960 hasta 2024. Incluye PIB, comercio exterior, inflación, mercado laboral, sector externo, finanzas públicas, demografía y desigualdad, con contexto por período presidencial.

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np

COUNTRY = 'COLOMBIA'
PATH_DATA = 'data/macro_economics_indicators_2026.csv'
HEIGHT = 800

# ── Paleta consistente por presidente (usada en TODOS los gráficos) ──────────
PRESIDENTES_ORDEN = [
    'ALBERTO LLERAS CAMARGO',
    'GUILLERMO LEÓN VALENCIA',
    'CARLOS LLERAS RESTREPO',
    'MISAEL PASTRANA BORRERO',
    'ALFONSO LÓPEZ MICHELSEN',
    'JULIO CÉSAR TURBAY AYALA',
    'BELISARIO BETANCUR',
    'VIRGILIO BARCO',
    'CÉSAR GAVIRIA',
    'ERNESTO SAMPER',
    'ANDRÉS PASTRANA',
    'ÁLVARO URIBE VÉLEZ',
    'JUAN MANUEL SANTOS',
    'IVÁN DUQUE',
    'GUSTAVO PETRO',
]

_PALETTE = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd',
    '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf',
    '#aec7e8', '#ffbb78', '#98df8a', '#f7b6d2', '#c5b0d5',
]
PRES_COLOR_MAP = {p: c for p, c in zip(PRESIDENTES_ORDEN, _PALETTE)}

COLOR_SEQ = _PALETTE  # backward compat

def add_trend(fig, x, y, degree=2, row=None, col=None):
    mask = ~np.isnan(y.astype(float))
    xf, yf = x[mask], y.astype(float)[mask]
    if len(xf) < degree + 1:
        return
    coeffs = np.polyfit(xf, yf, degree)
    trend = np.polyval(coeffs, xf)
    trace = go.Scatter(x=xf, y=trend, mode='lines',
                       line=dict(color='red', dash='dash'),
                       name='Tendencia', showlegend=False)
    if row is not None:
        fig.add_trace(trace, row=row, col=col)
    else:
        fig.add_trace(trace)


presidentes_lista = [
    {"year": y, "presidente": p}
    for p, (a, b) in [
        ("Alberto Lleras Camargo",    (1960, 1962)),
        ("Guillermo León Valencia",   (1963, 1966)),
        ("Carlos Lleras Restrepo",    (1967, 1970)),
        ("Misael Pastrana Borrero",   (1971, 1974)),
        ("Alfonso López Michelsen",   (1975, 1978)),
        ("Julio César Turbay Ayala",  (1979, 1982)),
        ("Belisario Betancur",        (1983, 1986)),
        ("Virgilio Barco",            (1987, 1990)),
        ("César Gaviria",             (1991, 1994)),
        ("Ernesto Samper",            (1995, 1998)),
        ("Andrés Pastrana",           (1999, 2002)),
        ("Álvaro Uribe Vélez",        (2003, 2010)),
        ("Juan Manuel Santos",        (2011, 2018)),
        ("Iván Duque",               (2019, 2022)),
        ("Gustavo Petro",             (2023, 2024)),
    ]
    for y in range(a, b + 1)
]

df_presidentes = pd.DataFrame(presidentes_lista)
df_presidentes['presidente'] = df_presidentes['presidente'].str.upper()


In [50]:
df_raw = pd.read_csv(PATH_DATA, sep=';', decimal=',')
df = df_raw[df_raw['country_name'] == COUNTRY].copy()

df = pd.merge(df, df_presidentes, on='year', how='left')

df = df[df['year'] <= 2024].reset_index(drop=True)

df['balance_comercial'] = df['exports_of_goods_and_services'] - df['imports_of_goods_and_services']

df = df.sort_values('year').reset_index(drop=True)

print(f"Período: {df['year'].min()} – {df['year'].max()}")
print(f"Registros: {len(df)} | Columnas: {df.shape[1]}")
df.head(3)

Período: 1960 – 2024
Registros: 65 | Columnas: 29


,country_code,region_name,sub_region_name,intermediate_region,country_name,income_group,year,total_gdp,total_gdp_million,gdp_variation,...,external_debt,external_debt_pct_gdp,deuda_publica,ingresos_tributarios,poblacion,gini,life_expectancy_women,life_expectancy_men,presidente,balance_comercial
0,COL,AMERICAS,LATIN AMERICA AND THE CARIBBEAN,SOUTH AMERICA,COLOMBIA,INGRESO MEDIANO ALTO,1960,4.031153e+09,4031.15,0.00,...,0.0,0.0,0.0,0.0,15606209.0,0.0,59.19,55.10,ALBERTO LLERAS CAMARGO,0.93
1,COL,AMERICAS,LATIN AMERICA AND THE CARIBBEAN,SOUTH AMERICA,COLOMBIA,INGRESO MEDIANO ALTO,1961,4.540448e+09,4540.45,5.09,...,0.0,0.0,0.0,0.0,16095203.0,0.0,59.77,55.71,ALBERTO LLERAS CAMARGO,-0.84
2,COL,AMERICAS,LATIN AMERICA AND THE CARIBBEAN,SOUTH AMERICA,COLOMBIA,INGRESO MEDIANO ALTO,1962,4.955544e+09,4955.54,5.41,...,0.0,0.0,0.0,0.0,16599029.0,0.0,60.31,56.31,ALBERTO LLERAS CAMARGO,-0.01


In [51]:
numeric_cols = [
    'total_gdp_million', 'gdp_variation', 'total_gdp_percapita', 'gdp_percapita_variation',
    'exports_of_goods_and_services', 'imports_of_goods_and_services', 'balance_comercial', 'cuenta_corriente',
    'inflation_rate', 'foreign_direct_investment', 'unemployment_rate',
    'international_reserves', 'external_debt', 'external_debt_pct_gdp',
    'deuda_publica', 'ingresos_tributarios',
    'poblacion', 'gini', 'life_expectancy_women', 'life_expectancy_men'
]

# Tabla de estadísticas descriptivas
desc = df[numeric_cols].describe().round(2)
desc.loc['missing'] = df[numeric_cols].isnull().sum()
desc

,total_gdp_million,gdp_variation,total_gdp_percapita,gdp_percapita_variation,exports_of_goods_and_services,imports_of_goods_and_services,balance_comercial,cuenta_corriente,inflation_rate,foreign_direct_investment,unemployment_rate,international_reserves,external_debt,external_debt_pct_gdp,deuda_publica,ingresos_tributarios,poblacion,gini,life_expectancy_women,life_expectancy_men
count,65.00,65.00,65.00,64.00,65.00,65.00,65.00,65.00,65.00,65.00,65.00,6.500000e+01,6.500000e+01,65.00,65.00,65.00,65.00,65.00,63.00,63.00
mean,121096.47,3.91,2812.34,6.05,15.31,17.08,-1.77,-2.13,13.80,2.01,6.04,1.556334e+10,4.280578e+10,28.41,19.39,4.56,33855828.91,23.98,71.73,65.51
std,130490.73,2.71,2570.20,10.77,2.41,3.94,3.47,2.61,9.18,1.77,6.21,1.934456e+10,5.442240e+10,15.03,30.72,6.70,11238651.39,26.97,6.20,5.19
min,4031.15,-7.19,258.30,-23.68,9.98,9.71,-7.66,-7.84,2.02,0.00,0.00,7.712566e+07,0.000000e+00,0.00,0.00,0.00,15606209.00,0.00,59.19,55.10
25%,15341.40,2.50,643.01,-1.81,13.52,13.80,-4.37,-4.20,5.81,0.45,0.00,1.291230e+09,4.123797e+09,22.73,0.00,0.00,23858810.00,0.00,66.97,62.16
50%,58418.99,4.09,1730.39,5.94,15.69,16.75,-1.58,-2.21,10.87,1.60,8.25,7.907620e+09,1.756082e+10,30.05,0.00,0.00,33760571.00,0.00,73.30,64.97
75%,232468.66,5.41,5250.97,15.80,16.77,20.67,0.62,0.00,22.53,3.38,11.06,2.367059e+10,4.703992e+10,38.64,52.54,13.03,43758808.00,53.50,76.92,69.74
max,418818.15,10.80,8279.10,25.86,20.39,27.89,6.85,4.78,33.80,7.03,20.52,6.189797e+10,2.017636e+11,57.65,91.16,17.56,52886363.00,58.50,79.72,73.84
missing,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.000000e+00,0.000000e+00,0.00,0.00,0.00,0.00,0.00,2.00,2.00


# Catálogo de Indicadores

| Variable | Descripción | Unidad | Cobertura real |
|---|---|---|---|
| `total_gdp_million` | PIB a precios corrientes | Millones USD | 1960–2024 |
| `gdp_variation` | Crecimiento anual del PIB | % | 1960–2024 |
| `total_gdp_percapita` | PIB per cápita a precios corrientes | USD | 1960–2024 |
| `gdp_percapita_variation` | Crecimiento del PIB per cápita | % | 1961–2024 |
| `exports_of_goods_and_services` | Exportaciones de bienes y servicios | % del PIB | 1960–2024 |
| `imports_of_goods_and_services` | Importaciones de bienes y servicios | % del PIB | 1960–2024 |
| `balance_comercial` | Exportaciones menos importaciones (calculada) | % del PIB | 1960–2024 |
| `cuenta_corriente` | Saldo en cuenta corriente | % del PIB | 1960–2024 |
| `inflation_rate` | Inflación anual (IPC) | % | 1960–2024 |
| `foreign_direct_investment` | Inversión extranjera directa neta | % del PIB | 1960–2024 |
| `unemployment_rate` | Tasa de desempleo | % | ~1991–2024 |
| `international_reserves` | Reservas internacionales brutas | USD corrientes | 1960–2024 |
| `external_debt` | Deuda externa total (pública + privada) | USD corrientes | ~1970–2024 |
| `external_debt_pct_gdp` | Deuda externa como porcentaje del PIB | % del PIB | ~1970–2024 |
| `deuda_publica` | Deuda bruta del Gobierno Central | % del PIB | ~2000–2024 |
| `ingresos_tributarios` | Recaudo tributario del Gobierno Central | % del PIB | ~2000–2024 |
| `poblacion` | Población total | Personas | 1960–2024 |
| `gini` | Coeficiente de Gini (desigualdad de ingreso) | 0–100 | ~1992–2024 (esporádico) |
| `life_expectancy_women` | Esperanza de vida al nacer, mujeres | Años | 1960–2022 |
| `life_expectancy_men` | Esperanza de vida al nacer, hombres | Años | 1960–2022 |

> **Fuente:** Dataset consolidado de indicadores macroeconómicos mundiales — `macro_economics_indicators_2026.csv`. Las series con cobertura parcial (desempleo, Gini, deuda pública) registran cero en años sin dato oficial disponible.


# Análisis Descriptivo Univariado

Distribución histórica de cada indicador: histograma con curva de densidad (KDE) y estadísticas clave (media, mediana, desviación estándar, sesgo). Los ceros que representan ausencia de datos son excluidos.


In [ ]:
from scipy import stats as scipy_stats

VARS_DESC = [
    ('total_gdp_million',              'PIB Total',                    'Millones USD'),
    ('gdp_variation',                  'Crecimiento PIB',              '% anual'),
    ('total_gdp_percapita',            'PIB per Cápita',               'USD'),
    ('gdp_percapita_variation',        'Crecimiento PIB per Cápita',   '% anual'),
    ('exports_of_goods_and_services',  'Exportaciones',                '% PIB'),
    ('imports_of_goods_and_services',  'Importaciones',                '% PIB'),
    ('balance_comercial',              'Balance Comercial',            '% PIB'),
    ('cuenta_corriente',               'Cuenta Corriente',             '% PIB'),
    ('inflation_rate',                 'Inflación (IPC)',               '% anual'),
    ('foreign_direct_investment',      'IED Neta',                     '% PIB'),
    ('unemployment_rate',              'Desempleo',                    '%'),
    ('international_reserves',         'Reservas Internacionales',     'USD'),
    ('external_debt',                  'Deuda Externa',                'USD'),
    ('external_debt_pct_gdp',          'Deuda Externa',                '% PIB'),
    ('deuda_publica',                  'Deuda Pública',                '% PIB'),
    ('ingresos_tributarios',           'Ingresos Tributarios',         '% PIB'),
    ('poblacion',                      'Población',                    'Personas'),
    ('gini',                           'Coeficiente Gini',             'Índice 0-100'),
    ('life_expectancy_women',          'Esperanza de Vida — Mujeres',  'Años'),
    ('life_expectancy_men',            'Esperanza de Vida — Hombres',  'Años'),
]

for col_name, label, unit in VARS_DESC:
    serie = df[col_name].replace(0, np.nan).dropna()
    if len(serie) < 5:
        continue

    media   = serie.mean()
    mediana = serie.median()
    std     = serie.std()
    skew    = serie.skew()

    # KDE
    kde_x = np.linspace(serie.min(), serie.max(), 300)
    kde_y = scipy_stats.gaussian_kde(serie)(kde_x)
    n_bins = min(20, max(5, len(serie) // 3))
    counts, edges = np.histogram(serie, bins=n_bins)
    bin_width = edges[1] - edges[0]
    kde_y_scaled = kde_y * len(serie) * bin_width

    fig = go.Figure()

    fig.add_trace(go.Bar(
        x=[(edges[i] + edges[i+1]) / 2 for i in range(len(counts))],
        y=counts, width=bin_width * 0.9,
        marker_color='steelblue', opacity=0.7,
        name='Frecuencia', showlegend=False,
    ))
    fig.add_trace(go.Scatter(
        x=kde_x, y=kde_y_scaled, mode='lines',
        line=dict(color='firebrick', width=2), name='Densidad (KDE)',
    ))

    fig.add_vline(x=media,   line_dash='dash', line_color='darkorange',
                  annotation_text=f'Media: {media:,.2f}',   annotation_position='top right')
    fig.add_vline(x=mediana, line_dash='dot',  line_color='darkgreen',
                  annotation_text=f'Mediana: {mediana:,.2f}', annotation_position='top left')

    fig.update_layout(
        title=(f'Distribución Histórica — {label} ({unit})<br>'
               f'<sup>n={len(serie)} | σ={std:,.2f} | Sesgo={skew:.2f} | '
               f'Min={serie.min():,.2f} | Max={serie.max():,.2f}</sup>'),
        xaxis_title=f'{label} ({unit})',
        yaxis_title='Frecuencia',
        height=450,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )
    fig.show()


# PIB — Producto Interno Bruto

In [52]:
## PIB total en millones USD y PIB per cápita
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(x=df['year'], y=df['total_gdp_million'],
                     name='PIB total (mill. USD)', marker_color='steelblue', opacity=0.7),
              secondary_y=False)

fig.add_trace(go.Scatter(x=df['year'], y=df['total_gdp_percapita'],
                          name='PIB per cápita (USD)', mode='lines+markers',
                          line=dict(color='darkorange', width=2)),
              secondary_y=True)

add_trend(fig, df['year'].values, df['total_gdp_million'].values, degree=2)

fig.update_layout(title='PIB Total y PIB per Cápita de Colombia (1960–2024)',
                  xaxis_title='Año', height=HEIGHT)
fig.update_yaxes(title_text='PIB Total (millones USD)', secondary_y=False)
fig.update_yaxes(title_text='PIB per Cápita (USD)', secondary_y=True)
fig.show()

In [53]:
## Variación del PIB y variación del PIB per cápita
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Crecimiento del PIB (% anual)',
                                    'Crecimiento del PIB per Cápita (% anual)'))

fig.add_trace(go.Bar(x=df['year'], y=df['gdp_variation'],
                     marker_color=np.where(df['gdp_variation'] >= 0, 'steelblue', 'tomato'),
                     name='Crecimiento PIB'), row=1, col=1)
add_trend(fig, df['year'].values, df['gdp_variation'].values, row=1, col=1)

fig.add_trace(go.Bar(x=df['year'], y=df['gdp_percapita_variation'],
                     marker_color=np.where(df['gdp_percapita_variation'].fillna(0) >= 0, 'mediumseagreen', 'salmon'),
                     name='Crecimiento PIB per cápita'), row=2, col=1)
add_trend(fig, df['year'].values, df['gdp_percapita_variation'].fillna(0).values, row=2, col=1)

fig.update_layout(title='Tasas de Crecimiento del PIB — Colombia', height=HEIGHT, showlegend=False)
fig.show()

In [ ]:
## PIB total por período presidencial
fig = px.bar(df, x='year', y='total_gdp_million', color='presidente',
             color_discrete_map=PRES_COLOR_MAP,
             category_orders={'presidente': PRESIDENTES_ORDEN})
add_trend(fig, df['year'].values, df['total_gdp_million'].values, degree=2)
fig.update_layout(title='PIB de Colombia por Período Presidencial',
                  xaxis_title='Año', yaxis_title='PIB (millones USD)', height=HEIGHT)
fig.show()


In [55]:
## Exportaciones e importaciones como % del PIB
fig = px.bar(df, x='year', y=['exports_of_goods_and_services', 'imports_of_goods_and_services'],
             barmode='group',
             labels={'value': '% del PIB', 'variable': 'Indicador'},
             color_discrete_map={'exports_of_goods_and_services': 'steelblue',
                                 'imports_of_goods_and_services': 'tomato'})

fig.for_each_trace(lambda t: t.update(name='Exportaciones' if 'exports' in t.name else 'Importaciones'))
fig.update_layout(title='Exportaciones e Importaciones de Colombia (% PIB)',
                  xaxis_title='Año', height=HEIGHT)
fig.show()# Comercio Exterior

In [56]:
# Balance comercial
fig = go.Figure()

colors_bc = np.where(df['balance_comercial'] >= 0, 'mediumseagreen', 'tomato')
fig.add_trace(go.Bar(x=df['year'], y=df['balance_comercial'],
                     marker_color=colors_bc, name='Balance comercial'))
add_trend(fig, df['year'].values, df['balance_comercial'].values)

fig.update_layout(title='Balance Comercial (Exportaciones − Importaciones, % PIB) — Colombia', height=HEIGHT, showlegend=False)
fig.show()

In [57]:
# Cuenta corriente
fig = go.Figure()

cc = df['cuenta_corriente'].fillna(0)
colors_cc = np.where(cc >= 0, 'steelblue', 'salmon')
fig.add_trace(go.Bar(x=df['year'], y=cc,
                     marker_color=colors_cc, name='Cuenta corriente'))
add_trend(fig, df['year'].values, cc.values)

fig.update_layout(title='Cuenta Corriente (% PIB) — Colombia', height=HEIGHT, showlegend=False)
fig.show()

In [ ]:
## Exportaciones e importaciones por período presidencial
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=('Exportaciones (% PIB)', 'Importaciones (% PIB)'))

seen = set()
for i, col_name in enumerate([('exports_of_goods_and_services'), ('imports_of_goods_and_services')], 1):
    for pres in PRESIDENTES_ORDEN:
        sub = df[df['presidente'] == pres]
        if sub.empty:
            continue
        show_leg = (i == 1 and pres not in seen)
        fig.add_trace(go.Bar(x=sub['year'], y=sub[col_name],
                             name=pres, marker_color=PRES_COLOR_MAP[pres],
                             showlegend=show_leg, legendgroup=pres), row=1, col=i)
        seen.add(pres)
    add_trend(fig, df['year'].values, df[col_name].values, degree=4, row=1, col=i)

fig.update_layout(title='Comercio Exterior de Colombia por Período Presidencial',
                  height=HEIGHT, barmode='stack')
fig.show()


In [59]:
## Inflación anual (IPC, %)
fig = go.Figure()

fig.add_trace(go.Bar(x=df['year'], y=df['inflation_rate'],
                     marker_color='indianred', name='Inflación'))
add_trend(fig, df['year'].values, df['inflation_rate'].values, degree=2)

fig.update_layout(title='Inflación Anual de Colombia (1960–2024)', height=HEIGHT)
fig.show()

In [ ]:
## Inflación por período presidencial
fig = go.Figure()
for pres in PRESIDENTES_ORDEN:
    sub = df[df['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Bar(x=sub['year'], y=sub['inflation_rate'],
                         name=pres, marker_color=PRES_COLOR_MAP[pres]))
fig.update_layout(title='Inflación por Período Presidencial — Colombia',
                  xaxis_title='Año', yaxis_title='Inflación (%)',
                  height=HEIGHT, barmode='stack')
fig.show()


# Mercado Laboral

In [ ]:
## Tasa de desempleo por período presidencial
fig = px.bar(df, x='year', y='unemployment_rate', color='presidente',
             color_discrete_map=PRES_COLOR_MAP,
             category_orders={'presidente': PRESIDENTES_ORDEN},
             labels={'unemployment_rate': 'Desempleo (%)', 'year': 'Año'})
add_trend(fig, df['year'].values, df['unemployment_rate'].values, degree=2)
fig.update_layout(title='Tasa de Desempleo de Colombia por Período Presidencial', height=HEIGHT)
fig.show()


# Inversión Extranjera y Sector Externo

In [62]:
## Inversión extranjera directa y reservas internacionales
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Inversión Extranjera Directa Neta (% PIB)',
                                    'Reservas Internacionales (USD corrientes)'))

fig.add_trace(go.Bar(x=df['year'], y=df['foreign_direct_investment'],
                     marker_color=np.where(df['foreign_direct_investment'] >= 0, 'steelblue', 'tomato'),
                     name='IED'), row=1, col=1)
add_trend(fig, df['year'].values, df['foreign_direct_investment'].values, row=1, col=1)

fig.add_trace(go.Scatter(x=df['year'], y=df['international_reserves'],
                          mode='lines+markers', name='Reservas',
                          line=dict(color='darkorange', width=2)), row=2, col=1)
add_trend(fig, df['year'].values, df['international_reserves'].values, degree=2, row=2, col=1)

fig.update_layout(title='Inversión Extranjera y Reservas Internacionales — Colombia',
                  height=HEIGHT, showlegend=False)
fig.show()

In [63]:
## Deuda externa: stock total y como % del PIB
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(go.Bar(x=df['year'], y=df['external_debt'],
                     name='Deuda externa (USD)', marker_color='steelblue', opacity=0.7),
              secondary_y=False)

fig.add_trace(go.Scatter(x=df['year'], y=df['external_debt_pct_gdp'],
                          name='Deuda externa (% PIB)', mode='lines+markers',
                          line=dict(color='crimson', width=2)),
              secondary_y=True)

add_trend(fig, df['year'].values, df['external_debt_pct_gdp'].values, degree=2)

fig.update_layout(title='Deuda Externa de Colombia: Valor Absoluto y % del PIB',
                  xaxis_title='Año', height=HEIGHT)
fig.update_yaxes(title_text='Deuda Externa (USD)', secondary_y=False)
fig.update_yaxes(title_text='Deuda Externa (% PIB)', secondary_y=True)
fig.show()

In [64]:
#Deuda pública e ingresos tributarios (% PIB)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Deuda Pública del Gobierno Central (% PIB)',
                                    'Ingresos Tributarios (% PIB)'))

fig.add_trace(go.Scatter(x=df['year'], y=df['deuda_publica'],
                          mode='lines+markers', name='Deuda pública',
                          line=dict(color='firebrick', width=2),
                          fill='tozeroy', fillcolor='rgba(178,34,34,0.15)'), row=1, col=1)
add_trend(fig, df['year'].values, df['deuda_publica'].fillna(0).values, row=1, col=1)

fig.add_trace(go.Scatter(x=df['year'], y=df['ingresos_tributarios'],
                          mode='lines+markers', name='Ingresos tributarios',
                          line=dict(color='darkgreen', width=2),
                          fill='tozeroy', fillcolor='rgba(0,100,0,0.15)'), row=2, col=1)
add_trend(fig, df['year'].values, df['ingresos_tributarios'].fillna(0).values, row=2, col=1)

fig.update_layout(title='Indicadores Fiscales de Colombia', height=HEIGHT, showlegend=False)
fig.show()

In [65]:
## Población total
df['poblacion_millones'] = df['poblacion'] / 1_000_000

fig = go.Figure()
fig.add_trace(go.Scatter(x=df['year'], y=df['poblacion_millones'],
                          mode='lines+markers', name='Población',
                          line=dict(color='teal', width=2),
                          fill='tozeroy', fillcolor='rgba(0,128,128,0.1)'))
add_trend(fig, df['year'].values, df['poblacion_millones'].values, degree=2)

fig.update_layout(title='Población Total de Colombia (millones de personas)',
                  xaxis_title='Año', yaxis_title='Millones de personas', height=HEIGHT)
fig.show()# Demografía y Bienestar Social

In [66]:
## Esperanza de vida: mujeres vs hombres
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['year'], y=df['life_expectancy_women'],
                          mode='lines+markers', name='Mujeres',
                          line=dict(color='deeppink', width=2)))
fig.add_trace(go.Scatter(x=df['year'], y=df['life_expectancy_men'],
                          mode='lines+markers', name='Hombres',
                          line=dict(color='royalblue', width=2)))

# Brecha como área sombreada
fig.add_trace(go.Scatter(
    x=pd.concat([df['year'], df['year'][::-1]]),
    y=pd.concat([df['life_expectancy_women'], df['life_expectancy_men'][::-1]]),
    fill='toself', fillcolor='rgba(128,128,128,0.15)',
    line=dict(color='rgba(255,255,255,0)'), name='Brecha', showlegend=True
))

fig.update_layout(title='Esperanza de Vida de Colombia — Mujeres vs Hombres',
                  xaxis_title='Año', yaxis_title='Años', height=HEIGHT)
fig.show()

# Análisis por Período Presidencial — Box Plots

Distribución de cada indicador clave desagregada por presidente. Permite comparar nivel central, dispersión y valores extremos entre mandatos.


In [ ]:
VARS_BOX = [
    ('total_gdp_million',              'PIB Total (Millones USD)'),
    ('gdp_variation',                  'Crecimiento PIB (% anual)'),
    ('total_gdp_percapita',            'PIB per Cápita (USD)'),
    ('inflation_rate',                 'Inflación (% anual)'),
    ('unemployment_rate',              'Desempleo (%)'),
    ('foreign_direct_investment',      'IED Neta (% PIB)'),
    ('exports_of_goods_and_services',  'Exportaciones (% PIB)'),
    ('imports_of_goods_and_services',  'Importaciones (% PIB)'),
    ('balance_comercial',              'Balance Comercial (% PIB)'),
    ('cuenta_corriente',               'Cuenta Corriente (% PIB)'),
    ('external_debt_pct_gdp',          'Deuda Externa (% PIB)'),
    ('life_expectancy_women',          'Esperanza de Vida — Mujeres (años)'),
    ('life_expectancy_men',            'Esperanza de Vida — Hombres (años)'),
]

for col_name, label in VARS_BOX:
    df_box = df[['presidente', col_name, 'year']].copy()
    df_box[col_name] = df_box[col_name].replace(0, np.nan)

    fig = go.Figure()
    for pres in PRESIDENTES_ORDEN:
        sub = df_box[df_box['presidente'] == pres][col_name].dropna()
        if sub.empty:
            continue
        fig.add_trace(go.Box(
            y=sub, name=pres,
            marker_color=PRES_COLOR_MAP[pres],
            boxmean='sd',
            line_width=1.5,
        ))

    fig.update_layout(
        title=f'{label} — Por Período Presidencial',
        yaxis_title=label,
        xaxis_title='Presidente',
        height=500,
        showlegend=False,
        xaxis=dict(tickangle=-30),
    )
    fig.show()


# Análisis de Correlaciones

Mapa de calor de correlaciones de Pearson entre todos los indicadores numéricos. Los valores próximos a +1 o -1 indican fuerte relación lineal.


In [ ]:
CORR_COLS = [
    'total_gdp_million', 'gdp_variation', 'total_gdp_percapita',
    'exports_of_goods_and_services', 'imports_of_goods_and_services',
    'balance_comercial', 'cuenta_corriente', 'inflation_rate',
    'foreign_direct_investment', 'unemployment_rate',
    'external_debt_pct_gdp', 'deuda_publica', 'ingresos_tributarios',
    'poblacion', 'gini', 'life_expectancy_women',
]
CORR_LABELS = [
    'PIB Total', 'Crecim. PIB', 'PIB per Cápita',
    'Exportaciones', 'Importaciones',
    'Balance Comercial', 'Cuenta Corriente', 'Inflación',
    'IED Neta', 'Desempleo',
    'Deuda Ext. %PIB', 'Deuda Pública', 'Ing. Tributarios',
    'Población', 'Gini', 'Esp. Vida Mujeres',
]

df_corr = df[CORR_COLS].replace(0, np.nan)
corr_matrix = df_corr.corr(method='pearson').round(2)
corr_matrix.columns = CORR_LABELS
corr_matrix.index   = CORR_LABELS

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=CORR_LABELS,
    y=CORR_LABELS,
    colorscale='RdBu',
    zmid=0,
    zmin=-1, zmax=1,
    text=corr_matrix.values,
    texttemplate='%{text:.2f}',
    textfont=dict(size=9),
    colorbar=dict(title='Pearson r'),
))

fig.update_layout(
    title='Matriz de Correlaciones de Pearson — Indicadores Macroeconómicos Colombia',
    height=720,
    xaxis=dict(tickangle=-45, tickfont=dict(size=10)),
    yaxis=dict(tickfont=dict(size=10)),
)
fig.show()


# Análisis Relacional — Scatter Plots

Relaciones entre indicadores clave coloreados por período presidencial. Permite identificar patrones estructurales y outliers macroeconómicos.


In [ ]:
SCATTER_PAIRS = [
    ('gdp_variation',         'inflation_rate',               'Crecimiento PIB (% anual)',   'Inflación (% anual)'),
    ('gdp_variation',         'unemployment_rate',            'Crecimiento PIB (% anual)',   'Desempleo (%)'),
    ('gdp_variation',         'foreign_direct_investment',    'Crecimiento PIB (% anual)',   'IED Neta (% PIB)'),
    ('inflation_rate',        'unemployment_rate',            'Inflación (% anual)',         'Desempleo (%)'),
    ('total_gdp_percapita',   'life_expectancy_women',        'PIB per Cápita (USD)',        'Esperanza de Vida Mujeres (años)'),
    ('total_gdp_percapita',   'gini',                         'PIB per Cápita (USD)',        'Coeficiente Gini'),
    ('foreign_direct_investment', 'external_debt_pct_gdp',   'IED Neta (% PIB)',            'Deuda Externa (% PIB)'),
    ('exports_of_goods_and_services', 'balance_comercial',   'Exportaciones (% PIB)',       'Balance Comercial (% PIB)'),
]

for x_col, y_col, x_label, y_label in SCATTER_PAIRS:
    df_sc = df[[x_col, y_col, 'presidente', 'year']].replace(0, np.nan).dropna()

    fig = go.Figure()
    for pres in PRESIDENTES_ORDEN:
        sub = df_sc[df_sc['presidente'] == pres]
        if sub.empty:
            continue
        fig.add_trace(go.Scatter(
            x=sub[x_col], y=sub[y_col],
            mode='markers+text',
            name=pres,
            marker=dict(color=PRES_COLOR_MAP[pres], size=10, opacity=0.85,
                        line=dict(width=0.5, color='white')),
            text=sub['year'].astype(str),
            textposition='top center',
            textfont=dict(size=8),
        ))

    # Línea de tendencia global
    x_all = df_sc[x_col].values
    y_all = df_sc[y_col].values
    if len(x_all) > 2:
        coeffs = np.polyfit(x_all, y_all, 1)
        x_line = np.linspace(x_all.min(), x_all.max(), 100)
        r = np.corrcoef(x_all, y_all)[0, 1]
        fig.add_trace(go.Scatter(
            x=x_line, y=np.polyval(coeffs, x_line),
            mode='lines', line=dict(color='black', dash='dash', width=1.5),
            name=f'Tendencia (r={r:.2f})', showlegend=True,
        ))

    fig.update_layout(
        title=f'{x_label} vs {y_label}',
        xaxis_title=x_label,
        yaxis_title=y_label,
        height=550,
        legend=dict(font=dict(size=9)),
    )
    fig.show()


# Análisis Fiscal Ampliado

Comparación del desempeño fiscal por presidente: deuda pública, ingresos tributarios y esfuerzo fiscal (relación recaudo/deuda).


In [ ]:
## Promedios fiscales por presidente
df_fiscal = df[df['deuda_publica'] > 0].copy()

pres_fiscal = (
    df_fiscal.groupby('presidente')[['deuda_publica', 'ingresos_tributarios']]
    .mean().round(2)
    .reindex([p for p in PRESIDENTES_ORDEN if p in df_fiscal['presidente'].unique()])
    .reset_index()
)
pres_fiscal['esfuerzo_fiscal'] = (
    pres_fiscal['ingresos_tributarios'] / pres_fiscal['deuda_publica'] * 100
).round(2)

colors_bar = [PRES_COLOR_MAP[p] for p in pres_fiscal['presidente']]

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=(
                        'Deuda Pública Promedio (% PIB)',
                        'Ingresos Tributarios Promedio (% PIB)',
                        'Esfuerzo Fiscal: Recaudo / Deuda × 100',
                    ))

fig.add_trace(go.Bar(x=pres_fiscal['presidente'], y=pres_fiscal['deuda_publica'],
                     marker_color=colors_bar, name='Deuda Pública', showlegend=False), row=1, col=1)

fig.add_trace(go.Bar(x=pres_fiscal['presidente'], y=pres_fiscal['ingresos_tributarios'],
                     marker_color=colors_bar, name='Ing. Tributarios', showlegend=False), row=2, col=1)

fig.add_trace(go.Bar(x=pres_fiscal['presidente'], y=pres_fiscal['esfuerzo_fiscal'],
                     marker_color=colors_bar, name='Esfuerzo Fiscal', showlegend=False), row=3, col=1)

fig.update_layout(title='Indicadores Fiscales Promedio por Período Presidencial — Colombia',
                  height=800, xaxis3=dict(tickangle=-30))
fig.show()


In [ ]:
## Reservas internacionales como meses de importaciones
df['reservas_meses_import'] = (
    df['international_reserves'] /
    (df['imports_of_goods_and_services'] / 100 * df['total_gdp'] / 12)
).replace([np.inf, -np.inf], np.nan)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Reservas Internacionales (USD)',
                                    'Cobertura: Meses de Importaciones'))

for pres in PRESIDENTES_ORDEN:
    sub = df[df['presidente'] == pres]
    fig.add_trace(go.Bar(x=sub['year'], y=sub['international_reserves'],
                         name=pres, marker_color=PRES_COLOR_MAP[pres],
                         showlegend=True, legendgroup=pres), row=1, col=1)
    fig.add_trace(go.Scatter(x=sub['year'], y=sub['reservas_meses_import'],
                             name=pres, marker_color=PRES_COLOR_MAP[pres],
                             mode='lines+markers', showlegend=False, legendgroup=pres), row=2, col=1)

fig.add_hline(y=3, line_dash='dash', line_color='red',
              annotation_text='Mínimo recomendado (3 meses)', row=2, col=1)

fig.update_layout(title='Reservas Internacionales y Cobertura de Importaciones — Colombia',
                  height=700)
fig.show()


# Desigualdad, Demografía y Bienestar

Análisis del Coeficiente de Gini, brecha de esperanza de vida por género, y crecimiento poblacional.


In [ ]:
## Coeficiente de Gini por período presidencial (solo años con dato)
df_gini = df[df['gini'] > 0].copy()

fig = go.Figure()
for pres in PRESIDENTES_ORDEN:
    sub = df_gini[df_gini['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(
        x=sub['year'], y=sub['gini'],
        mode='markers+lines',
        name=pres,
        marker=dict(color=PRES_COLOR_MAP[pres], size=10),
        line=dict(color=PRES_COLOR_MAP[pres], width=2),
    ))

add_trend(fig, df_gini['year'].values, df_gini['gini'].values, degree=2)

fig.update_layout(
    title='Coeficiente de Gini de Colombia por Período Presidencial<br>'
          '<sup>Solo años con dato disponible — Valores más altos = mayor desigualdad</sup>',
    xaxis_title='Año', yaxis_title='Gini (0=igualdad perfecta, 100=desigualdad máxima)',
    height=HEIGHT,
)
fig.show()


In [ ]:
## Brecha de esperanza de vida (mujeres - hombres) por año y presidente
df_ev = df[df['life_expectancy_women'] > 0].copy()
df_ev['brecha_ev'] = df_ev['life_expectancy_women'] - df_ev['life_expectancy_men']

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Esperanza de Vida por Género',
                                    'Brecha (Mujeres − Hombres, años)'))

for pres in PRESIDENTES_ORDEN:
    sub = df_ev[df_ev['presidente'] == pres]
    if sub.empty:
        continue
    fig.add_trace(go.Scatter(x=sub['year'], y=sub['life_expectancy_women'],
                             name=pres, marker_color=PRES_COLOR_MAP[pres],
                             mode='markers+lines', showlegend=True, legendgroup=pres,
                             line=dict(width=2)), row=1, col=1)
    fig.add_trace(go.Scatter(x=sub['year'], y=sub['life_expectancy_men'],
                             name=pres, marker_color=PRES_COLOR_MAP[pres],
                             mode='markers+lines', showlegend=False, legendgroup=pres,
                             line=dict(width=2, dash='dot')), row=1, col=1)
    fig.add_trace(go.Bar(x=sub['year'], y=sub['brecha_ev'],
                         name=pres, marker_color=PRES_COLOR_MAP[pres],
                         showlegend=False, legendgroup=pres), row=2, col=1)

fig.update_layout(
    title='Esperanza de Vida por Género y Brecha — Colombia<br>'
          '<sup>Línea sólida = Mujeres | Línea punteada = Hombres</sup>',
    height=700,
)
fig.show()

## Crecimiento poblacional anual
df['pop_growth_pct'] = df['poblacion'].pct_change() * 100

fig2 = go.Figure()
for pres in PRESIDENTES_ORDEN:
    sub = df[df['presidente'] == pres]
    fig2.add_trace(go.Bar(x=sub['year'], y=sub['pop_growth_pct'],
                          name=pres, marker_color=PRES_COLOR_MAP[pres]))
add_trend(fig2, df['year'].values[1:], df['pop_growth_pct'].dropna().values, degree=2)
fig2.update_layout(
    title='Crecimiento Poblacional Anual (%) — Colombia por Período Presidencial',
    xaxis_title='Año', yaxis_title='Crecimiento (% anual)',
    height=HEIGHT,
)
fig2.show()


# Dashboard Resumen — Indicadores Clave por Presidente

Promedios de los principales indicadores macroeconómicos por mandato presidencial en una sola vista.


In [ ]:
## Tabla resumen de promedios por presidente
RESUMEN_COLS = {
    'gdp_variation':               'Crecim. PIB (%)',
    'inflation_rate':              'Inflación (%)',
    'unemployment_rate':           'Desempleo (%)',
    'foreign_direct_investment':   'IED (% PIB)',
    'exports_of_goods_and_services': 'Exportac. (% PIB)',
    'balance_comercial':           'Bal. Comercial (% PIB)',
    'external_debt_pct_gdp':       'Deuda Ext. (% PIB)',
}

df_res = df.copy()
for c in RESUMEN_COLS:
    df_res[c] = df_res[c].replace(0, np.nan)

resumen = (
    df_res.groupby('presidente')[list(RESUMEN_COLS.keys())]
    .mean().round(2)
    .reindex(PRESIDENTES_ORDEN)
    .dropna(how='all')
    .rename(columns=RESUMEN_COLS)
)

# Heatmap de promedios normalizados
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
resumen_norm = pd.DataFrame(
    scaler.fit_transform(resumen.fillna(resumen.mean())),
    index=resumen.index,
    columns=resumen.columns,
)

# Texto con valores reales para mostrar en celdas
text_vals = resumen.fillna('—').astype(str).values

fig = go.Figure(go.Heatmap(
    z=resumen_norm.values,
    x=resumen_norm.columns.tolist(),
    y=resumen_norm.index.tolist(),
    colorscale='RdYlGn',
    zmid=0.5,
    text=text_vals,
    texttemplate='%{text}',
    textfont=dict(size=9),
    showscale=True,
    colorbar=dict(title='Valor<br>normalizado'),
))

fig.update_layout(
    title='Dashboard: Promedios Macroeconómicos por Presidente (normalizado 0–1)<br>'
          '<sup>Verde = mejor desempeño relativo | Rojo = peor desempeño relativo — varía por indicador</sup>',
    height=600,
    xaxis=dict(tickangle=-30, tickfont=dict(size=10)),
    yaxis=dict(tickfont=dict(size=10)),
)
fig.show()

print("\n=== Tabla de valores reales ===")
print(resumen.to_string())
